In [26]:
%pip install yfinance
%pip install matplotlib
%pip install pandas
%pip install numpy
%pip install oracledb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [27]:
# import modules
from datetime import datetime, timedelta
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import oracledb
import typing
import re

In [60]:
def get_fx_rate_for_day(base: str, quote: str, date_str: str) -> float | None:
   """
   Obtine cursul de schimb pentru o anumita data
   """
   
   target = pd.Timestamp(date_str)

   # Try direct pair first, then inverse as fallback
   candidates = [
      (f"{base}{quote}=X", False),
      (f"{quote}{base}=X", True),
   ]

   for symbol, invert in candidates:
      # Pull a small window to survive weekends/holidays
      start = (target - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
      end = (target + pd.Timedelta(days=2)).strftime("%Y-%m-%d")

      df: pd.DataFrame | None = None
      try:
         df = yf.download(symbol, start=start, end=end, interval="1d", progress=False)
      except Exception as e:
         print(f"Error downloading {symbol}: {e} - trying reverse pair" if not invert else f"Error downloading {symbol}: {e} - giving up")
         continue
      
      if df is None or df.empty:
         continue

      # Keep only rows up to target day, then take latest available close
      df = df.loc[df.index <= target]
      if df.empty:
         continue
      
      close_series: pd.Series = df["Close"].dropna().iloc[-1]
      close_val = float(close_series.values[0])
      
      if close_val <= 0:
         continue

      return (1.0 / close_val) if invert else close_val

   return None

# # KRW -> CHF on 2026-05-23
# # DKK -> CNY on 2026-05-03
# # CNY -> DKK on 2026-05-10

# get_fx_rate_for_day("KRW", "CHF", "2026-05-23") 

In [61]:
# DB Stuff
ORACLE_HOST = "10.19.49.10"
ORACLE_PORT = 1522
ORACLE_SERVICE = "XE"
ORACLE_USER = "proiect"
ORACLE_PASSWORD = "proiect"

def db_get_connection() -> oracledb.Connection:
   """Se conecteaza la baza de date si returneaza conexiunea

   Returns:
      oracledb.Connection: Conexiunea la baza de date Oracle
   """
   dsn = f"{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}"
   print(f"Connecting to Oracle DB with DSN: '{dsn}'")
   connection: oracledb.Connection = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=dsn)
   print(f"Connected to Oracle DB with DSN: '{dsn}'; user: '{ORACLE_USER}'")
   return connection

def db_run_sql(conn: oracledb.Connection, sql: str, params: typing.Optional[dict] = None, fetch: bool = False, commit: bool = False) -> typing.Optional[list]: 
   """Executa o interogare SQL pe baza de date

   Args:
      conn (oracledb.Connection): Conexiunea la baza de date
      sql (str): Interogarea SQL de executat
      params (dict, optional): Parametrii pentru interogare. Defaults to None.
      fetch (bool, optional): Daca True, returneaza rezultatele interogarii. Defaults to False.

   Returns:
      list: Rezultatele interogarii daca fetch este True, altfel None
   """
   sql_print = str(sql).replace('\n', ' ').replace('\t', ' ')
   if len(sql_print) > 100:
      sql_print = sql_print[:100] + '...'
   print(f"Running SQL query: fetch={fetch}, commit={commit}, SQL: \"{sql_print}\", params: {params}")
   cursor = conn.cursor()
   if params:
      cursor.execute(sql, params)
   else:
      cursor.execute(sql)
   
   if fetch:
      return cursor.fetchall()

   if commit:
      conn.commit()
   return None

oracle_conn: oracledb.Connection = db_get_connection()

Connecting to Oracle DB with DSN: '10.19.49.10:1522/XE'
Connected to Oracle DB with DSN: '10.19.49.10:1522/XE'; user: 'proiect'


In [ ]:
monede: list[str] = [ m[0] for m in db_run_sql(oracle_conn, "SELECT cod_moneda FROM moneda", fetch=True) ] # type: ignore
# select 10 random pairs from monede
monede_exchange: list[tuple[str, str]] = list(zip(np.random.choice(monede, size=len(monede), replace=False).tolist(), monede))
# drop pairs with same value (e.g. ('RON', 'RON'))
monede_exchange = [ (m1, m2) for m1, m2 in monede_exchange if m1 != m2 ]

# append monede + monede reversed to guarantee at least 10 pairs
for i in range(len(monede)):
   m1 = monede[i]
   m2 = monede[(i + 1) % len(monede)]
   monede_exchange.append((m1, m2))

monede_exchange = monede_exchange[:10] # take only first 10 pairs

monede_insert_sqls: list[str] = []
for m1, m2 in monede_exchange:
   # get a random date between 2026-04-01 and 2026-05-23
   random_date: np.datetime64 = np.random.choice(pd.date_range("2026-05-01", "2026-05-23"))
   random_date_str = random_date.astype(datetime).strftime('%Y-%m-%d')
   exchange_rate = get_fx_rate_for_day(m1, m2, random_date_str)
   
   ins_str = f"""
      INSERT INTO curs_valutar (cod_moneda_sursa, cod_moneda_destinatie, data_curs, valoare_curs)
      VALUES ('{m1}', '{m2}', TO_DATE('{random_date_str}', 'YYYY-MM-DD'), {exchange_rate});
   """.strip()
   ins_str = re.sub(r"\s+", " ", ins_str)
   monede_insert_sqls.append(ins_str)
print(f"Generated {len(monede_insert_sqls)} SQL insert statements for curs_valutar:")

for sql in monede_insert_sqls:
   print(sql)

Running SQL query: fetch=True, commit=False, SQL: "SELECT cod_moneda FROM moneda", params: None
Generated 10 SQL insert statements for curs_valutar:
INSERT INTO curs_valutar (cod_moneda_sursa, cod_moneda_destinatie, data_curs, valoare_curs) VALUES ('DKK', 'CHF', TO_DATE('2026-05-18', 'YYYY-MM-DD'), 0.1223360002040863)
INSERT INTO curs_valutar (cod_moneda_sursa, cod_moneda_destinatie, data_curs, valoare_curs) VALUES ('KRW', 'CNY', TO_DATE('2026-05-18', 'YYYY-MM-DD'), 0.004544000141322613)
INSERT INTO curs_valutar (cod_moneda_sursa, cod_moneda_destinatie, data_curs, valoare_curs) VALUES ('JPY', 'DKK', TO_DATE('2026-05-18', 'YYYY-MM-DD'), 0.04030900076031685)
INSERT INTO curs_valutar (cod_moneda_sursa, cod_moneda_destinatie, data_curs, valoare_curs) VALUES ('CNY', 'EUR', TO_DATE('2026-05-11', 'YYYY-MM-DD'), 0.12495800107717514)
INSERT INTO curs_valutar (cod_moneda_sursa, cod_moneda_destinatie, data_curs, valoare_curs) VALUES ('USD', 'GBP', TO_DATE('2026-05-16', 'YYYY-MM-DD'), 0.7467499971